<a href="https://colab.research.google.com/github/Ghala-Alotaibi/saudi-arabic-pragmatics-llm/blob/main/Evaluating_Pragmatic_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS421 Selected Topics in Computer Science — Experiment Notebook
**Group 4, Section 63** — King Faisal University, Department of Computer Science
Instructor: Dr. Hala Hamdoun

| Name | ID |
| --- | --- |
| Ghala Alotaibi | 223047156 |
| Salha Almarri | 223042287 |
| Haya Almuslim | 223034961 |

GitHub: https://github.com/Ghala-Alotaibi/saudi-arabic-pragmatics-llm

**Project:** *Evaluating Pragmatic Understanding of Saudi Arabic in Large Language Models* — Track 2 Comparative Study.

This notebook implements the full experiment designed in the Phase I report: dataset preparation, the shared zero-shot pipeline for all four candidate models, both evaluation tasks, accuracy tables, and the error analyses that feed into the Phase II write-up.

All runs use `SEED=42`, greedy decoding (temperature 0), identical Arabic prompt template, and the same answer-extraction method across all four models. Re-running reproduces the exact numbers reported.

## Setup

**Before running:** Runtime > Change runtime type > **A100 GPU** (or L4).

The first install cell pins `transformers==4.47.1` for Jais-13B compatibility — its 2023 custom modeling code is incompatible with `transformers` ≥ 5.

Place `ArPanEmo_test.csv` in `/content/` (upload via the file panel, or mount Drive — cell below).

In [ ]:
#Salha
!pip install -q "transformers==4.47.1"

Hugging Face login (LLaMA-3 and Jais need accepted access; ALLaM may too).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/Colab Notebooks/ArPanEmo_test.csv" /content/ArPanEmo_test.csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration, prompt, data, and answer extractor
Defines the four models, the shared Arabic system prompt and user template, the 18 pragmatics items (6 per phenomenon), the ArPanEmo loader (test split only, placeholder rows filtered), and the digit-extraction function used identically for every model.

In [ ]:
import re, os, json, random, gc, time
import pandas as pd

ARPANEMO_PATH = "ArPanEmo_test.csv"     # uploaded to /content in the Colab file panel
N_PER_LABEL   = 20                      # emotion items per label (11 labels -> ~220 items)
MAX_NEW_TOKENS = 8                      # same for every model
SEED = 42
OUT_DIR = "results"; os.makedirs(OUT_DIR, exist_ok=True)

# Phase I Table I. Baseline first.
MODELS = {
    "ALLaM-3rd-7B (baseline)": "ALLaM-AI/ALLaM-7B-Instruct-preview",
    "Jais-13B": "inception42/jais-13b-chat",
    "LLaMA-3-8B-Instruct":     "meta-llama/Meta-Llama-3-8B-Instruct",
    "Qwen2.5-7B-Instruct":     "Qwen/Qwen2.5-7B-Instruct",
}

SYSTEM_PROMPT = (
    "أنت خبير لغوي متخصص في تحليل النصوص العربية وفهم السياق والدلالات الضمنية "
    "في اللهجة السعودية. سأعرض عليك أسئلة اختيار من متعدد تتعلق بالفهم التداولي "
    "(المعنى المقصود خلف الكلام الظاهر).\n"
    "مهمتك:\n"
    "1. اقرأ السؤال والخيارات بعناية.\n"
    "2. حدد الإجابة الأصح بناءً على السياق والمعنى المقصود، وليس المعنى الحرفي فقط.\n"
    "3. أجب برقم الخيار الصحيح فقط (مثل: 1) دون أي شرح إضافي."
)
USER_TEMPLATE = "السؤال: {question}\nالخيارات:\n{options}\nالإجابة (رقم فقط):"

# ---- the 18 pragmatics items (6 sarcasm, 6 indirect speech acts, 6 idioms/proverbs) ----
TEST_ITEMS_RAW = [
 {
  "id": "sarcasm_01",
  "phenomenon": "sarcasm",
  "question": "بعد ما تأخر عنه صاحبه ساعتين، قال له: \"ما شاء الله عليك، دقيق مواعيد!\" -- وش يقصد؟",
  "options": [
   "1. يمدحه لأنه ملتزم بالمواعيد",
   "2. يعاتبه بشكل ساخر على تأخيره",
   "3. يسأله عن الساعة",
   "4. يشكره على الحضور"
  ],
  "answer": 2
 },
 {
  "id": "sarcasm_02",
  "phenomenon": "sarcasm",
  "question": "بعد ما خرب السيارة وهو يتعلم يسوق، قال له أبوه: \"ما شاء الله عليك سواق محترف!\" -- وش يقصد؟",
  "options": [
   "1. يمدحه فعلاً لأنه سواق ماهر",
   "2. يعاتبه بسخرية على قيادته السيئة",
   "3. يشجعه يتعلم أكثر",
   "4. يسأله عن رخصة القيادة"
  ],
  "answer": 2
 },
 {
  "id": "sarcasm_03",
  "phenomenon": "sarcasm",
  "question": "طلب منها تجهز الأكل بسرعة، وبعد ٣ ساعات قالت: \"تفضل، جاهز على السريع!\" فقال: \"ما شاء الله عليك، فعلاً سريعة!\" -- وش يقصد؟",
  "options": [
   "1. يمدح سرعتها",
   "2. يسخر من تأخرها الطويل",
   "3. يشكرها على الأكل",
   "4. يسألها عن الوقت"
  ],
  "answer": 2
 },
 {
  "id": "sarcasm_04",
  "phenomenon": "sarcasm",
  "question": "بعد ما نسي صديقه ميعاد عيد ميلاده، قال له: \"تسلم إيدك، ما نسيت ولا شي!\" -- وش يقصد؟",
  "options": [
   "1. يشكره لأنه تذكر",
   "2. يعاتبه بسخرية لأنه نسي",
   "3. يطلب منه هدية",
   "4. يهنئه بعيد ميلاده"
  ],
  "answer": 2
 },
 {
  "id": "sarcasm_05",
  "phenomenon": "sarcasm",
  "question": "طالب ما ذاكر للاختبار وطلعت درجته متدنية، قال له زميله: \"الله يعطيك العافية، تعبت من كثر المذاكرة!\" -- وش يقصد؟",
  "options": [
   "1. يمدح مجهوده بالمذاكرة",
   "2. يسخر من كسله وعدم مذاكرته",
   "3. يسأله عن نتيجة الاختبار",
   "4. يطلب يذاكر معه"
  ],
  "answer": 2
 },
 {
  "id": "sarcasm_06",
  "phenomenon": "sarcasm",
  "question": "بعد ما وصل الاجتماع متأخر نص ساعة، قالت له المديرة: \"يعطيك العافية على الالتزام بالوقت!\" -- وش تقصد؟",
  "options": [
   "1. تمدحه على التزامه",
   "2. تعاتبه بسخرية على تأخيره",
   "3. تسأله عن سبب الغياب",
   "4. تشكره على الحضور"
  ],
  "answer": 2
 },
 {
  "id": "indirect_01",
  "phenomenon": "indirect_speech_act",
  "question": "قالت الأم لابنها وهو جالس قدام التلفزيون: \"ما عندك واجبات اليوم؟\" -- وش تقصد؟",
  "options": [
   "1. تسأله سؤال عادي عن وجود واجبات",
   "2. تطلب منه بشكل غير مباشر يروح يذاكر",
   "3. تخبره إنه ما عليه واجبات",
   "4. تطلب منه يطفي التلفزيون بس"
  ],
  "answer": 2
 },
 {
  "id": "indirect_02",
  "phenomenon": "indirect_speech_act",
  "question": "قال الزوج لزوجته والصالة حارة جدًا: \"ودي أعرف ليش النافذة مفتوحة؟\" -- وش يقصد؟",
  "options": [
   "1. يسأل سؤال عادي عن سبب فتح النافذة",
   "2. يطلب منها بشكل غير مباشر تسكر النافذة أو تشغل المكيف",
   "3. يشتكي من الحر بدون أي طلب",
   "4. يطلب منها تفتح نافذة ثانية"
  ],
  "answer": 2
 },
 {
  "id": "indirect_03",
  "phenomenon": "indirect_speech_act",
  "question": "قالت المعلمة للطالب اللي يتكلم بالفصل: \"ما أدري ليش ما أقدر أكمل الشرح؟\" -- وش تقصد؟",
  "options": [
   "1. تسأل فعلاً عن سبب توقفها",
   "2. تطلب منه بشكل غير مباشر يسكت",
   "3. تشتكي من صعوبة الدرس",
   "4. تطلب منه يشرح بدالها"
  ],
  "answer": 2
 },
 {
  "id": "indirect_04",
  "phenomenon": "indirect_speech_act",
  "question": "قال الضيف لصاحب البيت: \"الجو برد شوي هنا مو؟\" -- وش يقصد؟",
  "options": [
   "1. يسأل عن درجة الحرارة بالخارج",
   "2. يطلب بشكل غير مباشر يسكرون المكيف أو يعطونه غطا",
   "3. يشتكي من الطقس العام",
   "4. يطلب يطلع يمشي"
  ],
  "answer": 2
 },
 {
  "id": "indirect_05",
  "phenomenon": "indirect_speech_act",
  "question": "قالت الأم وهي تشوف طبق الحلا فاضي: \"مين قاعد ياكل قبل الغدا؟\" -- وش تقصد؟",
  "options": [
   "1. تسأل سؤال فعلي عشان تعرف مين اللي أكل",
   "2. تعاتب بشكل غير مباشر اللي أكل الحلا قبل الوقت",
   "3. توزع الحلا على الكل",
   "4. تطلب حد يطبخ حلا جديد"
  ],
  "answer": 2
 },
 {
  "id": "indirect_06",
  "phenomenon": "indirect_speech_act",
  "question": "قال المدير للموظف اللي طلع متأخر: \"الساعة كم عندك؟\" -- وش يقصد؟",
  "options": [
   "1. يسأل فعلاً عن الوقت عشان ساعته وقفت",
   "2. يعاتبه بشكل غير مباشر على تأخيره عن الدوام",
   "3. يطلب منه يشتري له ساعة",
   "4. يسأل عن مواعيد الاجتماع القادم"
  ],
  "answer": 2
 },
 {
  "id": "idiom_01",
  "phenomenon": "idiom_proverb",
  "question": "قال له: \"اللي ما يعرف الصقر يشويه\" -- وش يقصد بهالمثل؟",
  "options": [
   "1. نصيحة بطريقة طبخ الصقر",
   "2. من يجهل قيمة الشيء الثمين يهدره أو يسيء التعامل معه",
   "3. وصف لنوع معين من الطيور",
   "4. تحذير من أكل الصقور"
  ],
  "answer": 2
 },
 {
  "id": "idiom_02",
  "phenomenon": "idiom_proverb",
  "question": "قال المعلم للطالب: \"اللي فات مات\" -- وش يقصد بهالمثل؟",
  "options": [
   "1. نصيحة بنسيان الماضي والتركيز على المستقبل",
   "2. الحديث عن شخص متوفي فعلاً",
   "3. وصف لحدث تاريخي",
   "4. تحذير من الموت"
  ],
  "answer": 1
 },
 {
  "id": "idiom_03",
  "phenomenon": "idiom_proverb",
  "question": "قالت الجدة: \"يد وحدة ما تصفق\" -- وش تقصد؟",
  "options": [
   "1. وصف حرفي لليد أثناء التصفيق",
   "2. التعاون والعمل الجماعي ضروري لتحقيق الأمور",
   "3. نصيحة بعدم التصفيق بيد وحدة",
   "4. وصف لحركة رياضية معينة"
  ],
  "answer": 2
 },
 {
  "id": "idiom_04",
  "phenomenon": "idiom_proverb",
  "question": "قال الأب لابنه: \"اللي يزرع يحصد\" -- وش يقصد؟",
  "options": [
   "1. نصيحة زراعية عن كيفية الزراعة",
   "2. من يجتهد ويعمل بيحصل على نتيجة جهوده لاحقًا",
   "3. وصف لموسم الحصاد",
   "4. تحذير من ترك الأرض بدون زراعة"
  ],
  "answer": 2
 },
 {
  "id": "idiom_05",
  "phenomenon": "idiom_proverb",
  "question": "قالت الأم: \"اللي ما له أول ما له تالي\" -- وش تقصد؟",
  "options": [
   "1. وصف لترتيب الأشياء",
   "2. من لا يهتم بأصله وجذوره ما راح يوصل لمستقبل جيد",
   "3. نصيحة بالبداية من الآخر",
   "4. وصف لقصة معينة"
  ],
  "answer": 2
 },
 {
  "id": "idiom_06",
  "phenomenon": "idiom_proverb",
  "question": "قال الجد: \"اللي يشوف بلوى غيره تهون عليه بلواه\" -- وش يقصد؟",
  "options": [
   "1. وصف حرفي لمشاهدة مشاكل الناس",
   "2. رؤية معاناة الآخرين تجعل الشخص يتقبل مشكلته بشكل أفضل",
   "3. نصيحة بعدم الاهتمام بمشاكل الآخرين",
   "4. تحذير من التدخل بمشاكل غيرك"
  ],
  "answer": 2
 }
]

def make_pragmatics_items(raw):
    """Strip baked-in '1. ' numbering and SHUFFLE option order (fixed seed, same for all models).
    Needed because 17 of the 18 original items had the correct answer at position 2,
    so an 'always answer 2' model would score ~94% without understanding anything."""
    out = []
    for it in raw:
        opts = [re.sub(r"^\s*\d+\.\s*", "", o) for o in it["options"]]
        gold_text = opts[it["answer"] - 1]
        rng = random.Random(f"{SEED}-{it['id']}"); rng.shuffle(opts)
        out.append({"id": it["id"], "phenomenon": it["phenomenon"], "question": it["question"],
                    "options": opts, "answer": opts.index(gold_text) + 1})
    return out

TEST_ITEMS = make_pragmatics_items(TEST_ITEMS_RAW)

# ---- ArPanEmo (secondary task): test split only, Twitter placeholder rows removed ----
def load_arpanemo(path, n_per_label):
    df = pd.read_csv(path)
    cols = {c.lower().strip(): c for c in df.columns}
    assert "post" in cols and "label" in cols, f"Expected 'post' and 'label' columns, found {list(df.columns)}"
    df = df.rename(columns={cols["post"]: "Post", cols["label"]: "Label"})
    df["Label"] = df["Label"].astype(str).str.strip().str.lower()
    df = df[df["Post"].astype(str).str.strip().ne("...") & (df["Post"].astype(str).str.len() > 3)]
    labels = sorted(df["Label"].unique())
    print(f"ArPanEmo: {len(df)} usable rows, {len(labels)} labels: {labels}")
    items = []
    for label in labels:
        sub = df[df["Label"] == label]
        for i, row in sub.sample(n=min(n_per_label, len(sub)), random_state=SEED).iterrows():
            rng = random.Random(f"{SEED}-emo{i}"); opts = labels.copy(); rng.shuffle(opts)
            items.append({"id": f"emotion_{i}", "phenomenon": "emotion_classification",
                          "question": f'ما هو الشعور الأساسي المعبر عنه في هذا المنشور؟\n\nالمنشور: "{str(row.Post).strip()}"',
                          "options": opts, "answer": opts.index(label) + 1, "gold_label": label})
    return items

EMOTION_ITEMS = load_arpanemo(ARPANEMO_PATH, N_PER_LABEL)

# ---- identical prompt + extractor for every model ----
ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def extract_choice(raw, n_options):
    """First 1-2 digit number, valid only if 1..n_options (10 and 11 are valid for ArPanEmo)."""
    m = re.search(r"\d{1,2}", raw.translate(ARABIC_DIGITS))
    if m and 1 <= int(m.group()) <= n_options:
        return int(m.group())
    return None

def user_text(item):
    return USER_TEMPLATE.format(question=item["question"],
                                options="\n".join(f"{i}. {o}" for i, o in enumerate(item["options"], 1)))

print(f"Pragmatics items: {len(TEST_ITEMS)} | Emotion items: {len(EMOTION_ITEMS)}")
print("Gold-position counts (pragmatics):", pd.Series([i["answer"] for i in TEST_ITEMS]).value_counts().sort_index().to_dict())

ArPanEmo: 255 usable rows, 11 labels: ['anger', 'anticipation', 'confusion', 'disgust', 'fear', 'happiness', 'neutral', 'optimism', 'pessimism', 'sadness', 'surprise']
Pragmatics items: 18 | Emotion items: 176
Gold-position counts (pragmatics): {1: 4, 2: 1, 3: 7, 4: 6}


## 2. Model runner (4-bit, greedy). Each model is loaded once and run on both tasks.

In [ ]:
#Ghala
import sys, types, torch
import transformers
import transformers.pytorch_utils as pu
import transformers.utils as tu

# 1. removed helper functions
def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
    mask = torch.ones(n_heads, head_size)
    heads = set(heads) - already_pruned_heads
    for head in heads:
        head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
        mask[head] = 0
    mask = mask.view(-1).contiguous().eq(1)
    index = torch.arange(len(mask))[mask].long()
    return heads, index

def _unused_prune(*a, **k):
    raise NotImplementedError("head pruning is not used in this experiment")

if not hasattr(pu, "find_pruneable_heads_and_indices"):
    pu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
for name in ["prune_conv1d_layer", "prune_linear_layer"]:
    if not hasattr(pu, name):
        setattr(pu, name, _unused_prune)

# 2. removed module
if "transformers.utils.model_parallel_utils" not in sys.modules:
    stub = types.ModuleType("transformers.utils.model_parallel_utils")
    stub.assert_device_map = lambda *a, **k: None
    stub.get_device_map = lambda *a, **k: {}
    sys.modules["transformers.utils.model_parallel_utils"] = stub
    tu.model_parallel_utils = stub

# 3. config defaults Jais's old config class expects
for attr, val in {"add_cross_attention": False, "is_encoder_decoder": False}.items():
    if not hasattr(transformers.PretrainedConfig, attr):
        setattr(transformers.PretrainedConfig, attr, val)

# 4. don't random-init quantized (byte) weights; re-runnable without recursion
if "normal_" in torch.Tensor.__dict__:
    del torch.Tensor.normal_
_orig_normal_ = torch.Tensor.normal_
def _safe_normal_(self, *a, **k):
    if self.dtype in (torch.uint8, torch.int8):
        return self
    return _orig_normal_(self, *a, **k)
torch.Tensor.normal_ = _safe_normal_

print("Jais patches applied")

Jais patches applied


##  3.Run all four models
Results are written to `results/all_results.json` after each model finishes. Re-running the cell skips models already in that file, so a crash only costs the current model.

In [ ]:
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          GenerationMixin, GenerationConfig)

def build_prompt(tok, item):
    """Chat template with system prompt; falls back to merging the system text into the user turn
    (models whose template has no system role), then to plain text (models with no chat template)."""
    u = user_text(item)
    for msgs in ([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": u}],
                 [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + u}]):
        try:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False), False
        except Exception:
            pass
    return SYSTEM_PROMPT + "\n\n" + u + "\n", True

def run_model(repo_id, task_items):
    tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        repo_id, device_map="auto", trust_remote_code=True,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16))
    model.eval()
    # old custom models (Jais) on new transformers: restore generate + generation_config
    if not hasattr(model, "generate"):
        model.__class__ = type(model.__class__.__name__, (model.__class__, GenerationMixin), {})
    if getattr(model, "generation_config", None) is None:
        model.generation_config = GenerationConfig.from_model_config(model.config)
    pad_id = tok.eos_token_id if tok.eos_token_id is not None else tok.pad_token_id

    def generate(item):
        prompt, plain = build_prompt(tok, item)
        enc = tok(prompt, return_tensors="pt", add_special_tokens=plain, return_token_type_ids=False)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        kw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=pad_id)   # temperature 0
        with torch.no_grad():
            try:
                out = model.generate(**enc, **kw)
            except Exception:
                out = model.generate(**enc, use_cache=False, **kw)   # same greedy output, avoids old cache code
        raw = tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True)
        return raw.split("</think>")[-1].strip()

    results = {}
    for task, items in task_items.items():
        rows = []
        for it in items:
            raw = generate(it); pred = extract_choice(raw, len(it["options"]))
            rows.append({"id": it["id"], "phenomenon": it["phenomenon"], "gold": it["answer"], "pred": pred,
                         "correct": pred == it["answer"], "raw_output": raw,
                         "gold_text": it["options"][it["answer"] - 1],
                         "pred_text": it["options"][pred - 1] if pred else None,
                         "question": it["question"]})
        results[task] = rows
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return results

def report(name, task, rows):
    c = sum(r["correct"] for r in rows)
    print(f"  {name} / {task}: {c}/{len(rows)} = {c/len(rows):.1%}  (unparsed: {sum(r['pred'] is None for r in rows)})")

from transformers import PreTrainedModel

# we never pass a head mask, so [None] per layer is the correct no-op
if not hasattr(PreTrainedModel, "get_head_mask"):
    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        return [None] * num_hidden_layers
    PreTrainedModel.get_head_mask = get_head_mask

if not hasattr(PreTrainedModel, "warn_if_padding_and_no_attention_mask"):
    PreTrainedModel.warn_if_padding_and_no_attention_mask = lambda self, *a, **k: None

if not hasattr(PreTrainedModel, "get_extended_attention_mask"):
    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        dtype = dtype or torch.float16
        ext = attention_mask[:, None, None, :].to(dtype=dtype)
        return (1.0 - ext) * torch.finfo(dtype).min
    PreTrainedModel.get_extended_attention_mask = get_extended_attention_mask

In [ ]:
import transformers
transformers.logging.set_verbosity_error()

RESULTS_PATH = f"{OUT_DIR}/all_results.json"
all_results = json.load(open(RESULTS_PATH, encoding="utf-8")) if os.path.exists(RESULTS_PATH) else {}

for name, repo in MODELS.items():
    if name in all_results:
        print("skip (already done):", name); continue
    print(f"\n=== {name} ({repo}) ===")
    t0 = time.time()
    try:
        res = run_model(repo, {"pragmatics": TEST_ITEMS, "emotion": EMOTION_ITEMS})
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}"); gc.collect(); torch.cuda.empty_cache(); continue
    all_results[name] = res
    for task, rows in res.items(): report(name, task, rows)
    json.dump(all_results, open(RESULTS_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"  done in {time.time()-t0:.0f}s")

skip (already done): ALLaM-3rd-7B (baseline)

=== Jais-13B (inception42/jais-13b-chat) ===


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (13653) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


  Jais-13B / pragmatics: 11/18 = 61.1%  (unparsed: 0)
  Jais-13B / emotion: 40/176 = 22.7%  (unparsed: 0)
  done in 128s
skip (already done): LLaMA-3-8B-Instruct
skip (already done): Qwen2.5-7B-Instruct


## 4. Accuracy tables (two tasks reported separately: chance 25% vs about 9%)

In [ ]:
#Haya
def acc_table(task):
    recs = []
    for model, res in all_results.items():
        rows = res[task]
        rec = {"model": model, "overall_%": round(100 * sum(r["correct"] for r in rows) / len(rows), 1),
               "n": len(rows), "unparsed": sum(r["pred"] is None for r in rows)}
        if task == "pragmatics":
            for ph in sorted({r["phenomenon"] for r in rows}):
                sub = [r for r in rows if r["phenomenon"] == ph]
                rec[ph + "_%"] = round(100 * sum(r["correct"] for r in sub) / len(sub), 1)
        recs.append(rec)
    return pd.DataFrame(recs).set_index("model")

print("=== Pragmatics (4 options, chance = 25%) ===")
df_prag = acc_table("pragmatics"); display(df_prag); df_prag.to_csv(f"{OUT_DIR}/accuracy_pragmatics.csv")
print("\n=== ArPanEmo emotion (11 options, chance ~ 9%) ===")
df_emo = acc_table("emotion"); display(df_emo); df_emo.to_csv(f"{OUT_DIR}/accuracy_emotion.csv")

# Position-bias check: does a model just keep picking the same option number?
print("\nPredicted option distribution on pragmatics (a spike on one number = position bias):")
display(pd.DataFrame({m: pd.Series([r["pred"] for r in res["pragmatics"]]).value_counts(dropna=False)
                      for m, res in all_results.items()}).fillna(0).astype(int).sort_index())

=== Pragmatics (4 options, chance = 25%) ===


,overall_%,n,unparsed,idiom_proverb_%,indirect_speech_act_%,sarcasm_%
model,,,,,,
ALLaM-3rd-7B (baseline),88.9,18,0,100.0,100.0,66.7
LLaMA-3-8B-Instruct,44.4,18,0,33.3,50.0,50.0
Qwen2.5-7B-Instruct,83.3,18,0,100.0,100.0,50.0
Jais-13B,61.1,18,0,83.3,83.3,16.7



=== ArPanEmo emotion (11 options, chance ~ 9%) ===


,overall_%,n,unparsed
model,,,
ALLaM-3rd-7B (baseline),54.0,176,0
LLaMA-3-8B-Instruct,31.2,176,0
Qwen2.5-7B-Instruct,43.2,176,0
Jais-13B,22.7,176,0



Predicted option distribution on pragmatics (a spike on one number = position bias):


,ALLaM-3rd-7B (baseline),LLaMA-3-8B-Instruct,Qwen2.5-7B-Instruct,Jais-13B
1,4,6,3,1
2,3,5,3,7
3,6,5,5,7
4,5,2,7,3


## 5. Error analysis
**ArPanEmo:** every wrong answer is classified same-valence or cross-valence, counted per model.

In [ ]:
VALENCE = {"happiness": "positive", "optimism": "positive",
           "anger": "negative", "disgust": "negative", "fear": "negative", "sadness": "negative", "pessimism": "negative",
           "anticipation": "neutral", "confusion": "neutral", "surprise": "neutral", "neutral": "neutral"}
# NOTE: anticipation/surprise/confusion are placed in 'neutral' (valence-ambiguous). Change here if the team decides otherwise.

recs = []
for model, res in all_results.items():
    same = cross = unparsed = 0
    for r in res["emotion"]:
        if r["correct"]: continue
        if r["pred_text"] is None: unparsed += 1; continue
        g, p = VALENCE.get(r["gold_text"]), VALENCE.get(r["pred_text"])
        if g is None or p is None: print("unmapped label:", r["gold_text"], r["pred_text"]); continue
        if g == p: same += 1
        else: cross += 1
    tot = same + cross
    recs.append({"model": model, "same_valence": same, "cross_valence": cross, "unparsed": unparsed,
                 "cross_%_of_errors": round(100 * cross / tot, 1) if tot else None})
df_val = pd.DataFrame(recs).set_index("model"); display(df_val); df_val.to_csv(f"{OUT_DIR}/emotion_valence_errors.csv")

,same_valence,cross_valence,unparsed,cross_%_of_errors
model,,,,
ALLaM-3rd-7B (baseline),33,48,0,59.3
LLaMA-3-8B-Instruct,52,69,0,57.0
Qwen2.5-7B-Instruct,47,53,0,53.0
Jais-13B,55,81,0,59.6


**Pragmatics:** every item a model got wrong, with the model's chosen option next to the gold one. Use this for the qualitative error categories (literal-meaning confusion, wrong-polarity, off-topic).

In [ ]:
wrong = []
for model, res in all_results.items():
    for r in res["pragmatics"]:
        if not r["correct"]:
            wrong.append({"model": model, "id": r["id"], "phenomenon": r["phenomenon"],
                          "gold_option": r["gold_text"], "model_chose": r["pred_text"], "raw": r["raw_output"]})
df_wrong = pd.DataFrame(wrong)
df_wrong.to_csv(f"{OUT_DIR}/pragmatics_wrong_items.csv", index=False, encoding="utf-8-sig")
print(len(df_wrong), "wrong answers across all models")
display(df_wrong.groupby(["model", "phenomenon"]).size().unstack(fill_value=0))

# Items that most models miss = hardest items
print("\nItems missed by the most models:")
display(df_wrong.groupby(["id", "phenomenon"]).size().sort_values(ascending=False).head(10).to_frame("models_wrong"))
df_wrong.head(30)

22 wrong answers across all models


phenomenon,idiom_proverb,indirect_speech_act,sarcasm
model,,,
ALLaM-3rd-7B (baseline),0,0,2
Jais-13B,1,1,5
LLaMA-3-8B-Instruct,4,3,3
Qwen2.5-7B-Instruct,0,0,3



Items missed by the most models:


,,models_wrong
id,phenomenon,
sarcasm_03,sarcasm,4
sarcasm_06,sarcasm,3
indirect_04,indirect_speech_act,2
sarcasm_01,sarcasm,2
idiom_05,idiom_proverb,2
sarcasm_02,sarcasm,2
idiom_06,idiom_proverb,1
idiom_03,idiom_proverb,1
idiom_02,idiom_proverb,1


,model,id,phenomenon,gold_option,model_chose,raw
0,ALLaM-3rd-7B (baseline),sarcasm_03,sarcasm,يسخر من تأخرها الطويل,يمدح سرعتها,2
1,ALLaM-3rd-7B (baseline),sarcasm_06,sarcasm,تعاتبه بسخرية على تأخيره,تمدحه على التزامه,2
2,LLaMA-3-8B-Instruct,sarcasm_02,sarcasm,يعاتبه بسخرية على قيادته السيئة,يشجعه يتعلم أكثر,3
3,LLaMA-3-8B-Instruct,sarcasm_03,sarcasm,يسخر من تأخرها الطويل,يمدح سرعتها,2
4,LLaMA-3-8B-Instruct,sarcasm_06,sarcasm,تعاتبه بسخرية على تأخيره,تشكره على الحضور,3
5,LLaMA-3-8B-Instruct,indirect_01,indirect_speech_act,تطلب منه بشكل غير مباشر يروح يذاكر,تسأله سؤال عادي عن وجود واجبات,1
6,LLaMA-3-8B-Instruct,indirect_03,indirect_speech_act,تطلب منه بشكل غير مباشر يسكت,تشتكي من صعوبة الدرس,2
7,LLaMA-3-8B-Instruct,indirect_04,indirect_speech_act,يطلب بشكل غير مباشر يسكرون المكيف أو يعطونه غطا,يسأل عن درجة الحرارة بالخارج,2
8,LLaMA-3-8B-Instruct,idiom_02,idiom_proverb,نصيحة بنسيان الماضي والتركيز على المستقبل,تحذير من الموت,2
9,LLaMA-3-8B-Instruct,idiom_03,idiom_proverb,التعاون والعمل الجماعي ضروري لتحقيق الأمور,نصيحة بعدم التصفيق بيد وحدة,1


## 6. Save everything
Download the `results/` folder from the Colab file panel (right-click > Download) before the session ends.

In [ ]:
!zip -qr results.zip results && ls -la results results.zip

-rw-r--r-- 1 root root 61354 Nov 15 20:02 results.zip

results:
total 408
drwxr-xr-x 2 root root   4096 Nov 15 20:01 .
drwxr-xr-x 1 root root   4096 Nov 15 20:02 ..
-rw-r--r-- 1 root root    144 Nov 15 20:00 accuracy_emotion.csv
-rw-r--r-- 1 root root    252 Nov 15 20:00 accuracy_pragmatics.csv
-rw-r--r-- 1 root root 391635 Nov 15 19:59 all_results.json
-rw-r--r-- 1 root root    185 Nov 15 20:01 emotion_valence_errors.csv
-rw-r--r-- 1 root root   3115 Nov 15 20:01 pragmatics_wrong_items.csv
